# 05 -- Portfolio Construction
Composite alpha from IC-significant factors -> four long/short variants.
Saves `portfolio_weights_monthly.parquet`.

In [1]:
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

factors  = pd.read_parquet('../data/processed/factors_monthly.parquet')
ic_df    = pd.read_csv('../data/processed/factor_ic_results.csv')
clusters = pd.read_parquet('../data/processed/network_clusters_monthly.parquet')

factors['date']  = pd.to_datetime(factors['date'])
clusters['date'] = pd.to_datetime(clusters['date'])

FACTOR_COLS  = [
    'mom_1m', 'mom_3m', 'mom_6m', 'vol_30d',
    'fees_mc', 'rev_mc', 'fees_growth_30d', 'rev_growth_30d',
    'dau_growth_30d', 'txns_growth_30d', 'dau_zscore',
    'tvl_mc', 'tvl_growth_30d',
]
MOM_FACTORS  = ['mom_1m', 'mom_3m', 'mom_6m', 'vol_30d']
FUND_FACTORS = [f for f in FACTOR_COLS if f not in MOM_FACTORS]

# Significant factors from NB03: t-stat > 1.0 (positive IC, significant)
# Factors with negative t-stat are excluded from the positive-IC composite;
# they are predictive but their z-score would need to be sign-flipped, which
# we handle instead via explicit vol_30d inversion below.
sig_pos  = ic_df[ic_df['t-stat'] > 1.0]['Factor'].tolist()
sig_mom  = [f for f in MOM_FACTORS  if f in sig_pos]
sig_fund = [f for f in FUND_FACTORS if f in sig_pos]

# For factors with strongly negative IC (|t| > 1.0, mean IC < 0), flip their
# sign so "higher score = better expected return" for the composite.
neg_sig  = ic_df[(ic_df['t-stat'] < -1.0)]['Factor'].tolist()
print(f"Positive-IC significant factors (t>1) : {sig_pos}")
print(f"  Momentum subset                     : {sig_mom}")
print(f"  Fund/usage subset                   : {sig_fund}")
print(f"Negative-IC significant factors (t<-1): {neg_sig}  [sign will be flipped]")


Positive-IC significant factors (t>1) : ['dau_growth_30d']
  Momentum subset                     : []
  Fund/usage subset                   : ['dau_growth_30d']
Negative-IC significant factors (t<-1): ['vol_30d']  [sign will be flipped]


In [2]:
## Merge cluster rank and standardise it cross-sectionally

factors_full = factors.merge(
    clusters[['symbol', 'date', 'within_cluster_rank']],
    on=['symbol', 'date'], how='left'
)
factors_full['within_cluster_rank'] = factors_full['within_cluster_rank'].fillna(0.5)

# Cross-sectional standardize within_cluster_rank (same procedure as factors)
col = 'within_cluster_rank'
for t, idx in factors_full.groupby('date').groups.items():
    s = factors_full.loc[idx, col].astype(float)
    if s.notna().sum() < 3:
        factors_full.loc[idx, col] = 0.0
        continue
    q01, q99 = s.quantile(0.01), s.quantile(0.99)
    s = s.clip(q01, q99)
    mu, sigma = s.mean(), s.std()
    if sigma > 1e-10:
        s = (s - mu) / sigma
    else:
        s = pd.Series(0.0, index=s.index)
    factors_full.loc[idx, col] = s.fillna(0.0).values

print(f"factors_full shape: {factors_full.shape}")
print(f"Columns: {list(factors_full.columns)}")


factors_full shape: (2400, 16)
Columns: ['symbol', 'date', 'mom_1m', 'mom_3m', 'mom_6m', 'vol_30d', 'fees_mc', 'rev_mc', 'fees_growth_30d', 'rev_growth_30d', 'dau_growth_30d', 'txns_growth_30d', 'dau_zscore', 'tvl_mc', 'tvl_growth_30d', 'within_cluster_rank']


In [3]:
## Build long/short weights for four variants

N_LONG  = 10   # top quintile of 50-coin universe
N_SHORT = 10   # bottom quintile

# Flip sign of strongly-negative-IC factors so higher score = better return
factors_cs = factors_full.copy()
for f in neg_sig:
    if f in factors_cs.columns:
        factors_cs[f] = -factors_cs[f]

# Fallback to full set (sign-adjusted) when no factor passes the positive-IC threshold
all_sig = sig_pos + neg_sig   # include sign-flipped negatives too
v1_factors = sig_mom  if sig_mom  else MOM_FACTORS
v2_factors = sig_fund if sig_fund else FUND_FACTORS
v3_factors = all_sig  if all_sig  else FACTOR_COLS
v4_factors = v3_factors + ['within_cluster_rank']

VARIANTS = {
    'v1_momentum':     v1_factors,
    'v2_fund_usage':   v2_factors,
    'v3_full':         v3_factors,
    'v4_full_cluster': v4_factors,
}

print("Variant factor sets (negative-IC factors sign-flipped):")
for k, v in VARIANTS.items():
    print(f"  {k:20s}: {v}")

all_weights = []

for variant, sel_factors in VARIANTS.items():
    for t, grp in factors_cs.groupby('date'):
        grp = grp.reset_index(drop=True)
        available = [f for f in sel_factors if f in grp.columns]
        composite = grp[available].mean(axis=1) if available else pd.Series(0.0, index=grp.index)
        composite.index = grp['symbol'].values

        long_syms  = composite.nlargest(N_LONG).index.tolist()
        short_syms = composite.nsmallest(N_SHORT).index.tolist()

        for sym in grp['symbol']:
            if sym in long_syms:
                w = 1.0 / N_LONG
            elif sym in short_syms:
                w = -1.0 / N_SHORT
            else:
                w = 0.0
            all_weights.append({'symbol': sym, 'date': t, 'weight': w, 'variant': variant})

weights_df = pd.DataFrame(all_weights)
print(f"\nWeights shape: {weights_df.shape}")

# Turnover estimate
print("\nAverage monthly turnover per variant (fraction of portfolio that changes):")
for variant in VARIANTS:
    v_df = weights_df[weights_df['variant'] == variant].pivot_table(
        index='date', columns='symbol', values='weight', fill_value=0)
    turnovers = v_df.diff().abs().sum(axis=1) / 2
    print(f"  {variant:22s}: {turnovers.mean():.2f}")


Variant factor sets (negative-IC factors sign-flipped):
  v1_momentum         : ['mom_1m', 'mom_3m', 'mom_6m', 'vol_30d']
  v2_fund_usage       : ['dau_growth_30d']
  v3_full             : ['dau_growth_30d', 'vol_30d']
  v4_full_cluster     : ['dau_growth_30d', 'vol_30d', 'within_cluster_rank']



Weights shape: (9600, 4)

Average monthly turnover per variant (fraction of portfolio that changes):
  v1_momentum           : 1.58
  v2_fund_usage         : 1.24
  v3_full               : 1.17
  v4_full_cluster       : 1.37


In [4]:
out_path = '../data/processed/portfolio_weights_monthly.parquet'
weights_df.to_parquet(out_path, index=False)
print(f"Saved: {out_path}  shape={weights_df.shape}")
print("\n>> Portfolio construction complete.")


Saved: ../data/processed/portfolio_weights_monthly.parquet  shape=(9600, 4)

>> Portfolio construction complete.
